# BYOD: Build Your Own DLM

Convert an autoregressive instruction model into a masked-diffusion model on a single GPU. **Quick model** keeps the complete 25,000-update recipe but uses a smaller backbone and rank-128 LoRA. **Paper reproduction** offers all four original 7B–9B models with rank-1024 LoRA.

## 1. Install the training code
The default setup installs the training package and notebook widgets. It skips Transformer Engine and bitsandbytes: **Quick model** uses ordinary FP16 LoRA training, not FP8 or 4-bit quantization. Installation output and elapsed times are shown below. Existing compatible PyTorch packages are reused.

If an earlier installation is still running, stop that cell before rerunning setup. After selecting a run mode, the separate FP8 step only installs Transformer Engine for paper mode on a supported GPU.


In [ ]:
import importlib
from importlib import metadata
import os, subprocess, sys, time
from pathlib import Path

REPOSITORY_URL = "https://github.com/RuurdKuiper/BYOD.git"
REPOSITORY_DIR = Path("/content/BYOD")

def setup_step(label, command):
    print(f"{label}…", flush=True)
    started = time.monotonic()
    subprocess.run(command, check=True)
    print(f"{label} completed in {time.monotonic() - started:.1f}s", flush=True)

if not (REPOSITORY_DIR / ".git").exists():
    setup_step("Clone BYOD", ["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)])
else:
    setup_step("Update BYOD", ["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only"])
os.chdir(REPOSITORY_DIR)
setup_step("Install training package and widgets", [
    sys.executable, "-m", "pip", "install", "--prefer-binary",
    "--upgrade-strategy", "only-if-needed", "-e", ".[colab]",
])

# PEFT can reject an old preinstalled TorchAO even when quantization is disabled.
# BYOD does not need TorchAO; only remove the known-incompatible versions.
from packaging.version import Version
try:
    torchao_version = metadata.version("torchao")
except metadata.PackageNotFoundError:
    torchao_version = None
if torchao_version is not None and Version(torchao_version) < Version("0.16.0"):
    setup_step(f"Remove incompatible optional TorchAO {torchao_version}", [
        sys.executable, "-m", "pip", "uninstall", "-y", "torchao",
    ])
else:
    print("No old TorchAO compatibility cleanup needed.", flush=True)

# Editable installs create a .pth file that this already-running kernel may
# not have processed. Make this checkout importable immediately.
for path in (REPOSITORY_DIR, REPOSITORY_DIR / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
importlib.invalidate_caches()
import diffusion_lm
print("BYOD imports from:", diffusion_lm.__file__, flush=True)
print("Installation complete. FP8 and bitsandbytes were not requested.", flush=True)


## 2. Connect Hugging Face
Add `HF_TOKEN` under Colab **Secrets** or sign in when prompted. The token only needs read access. You must first accept the selected model's license on Hugging Face.

In [ ]:
import os
from huggingface_hub import login

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

if token:
    os.environ["HF_TOKEN"] = token
    login(token=token, add_to_git_credential=False)
else:
    login(add_to_git_credential=False)

## 3. Choose a model and run mode
Start with **Quick model** for a lower-memory, full-length run. Select **Paper reproduction** only on a high-memory GPU. Ministral is available only in paper mode because there is no comparable official small text-only Ministral checkpoint.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

QUICK_MODELS = [
    ("Llama 3.2 1B Instruct", "llama"),
    ("Gemma 3 1B IT", "gemma"),
    ("Qwen2.5 1.5B Instruct", "qwen"),
]
PAPER_MODELS = [
    ("Llama 3.1 8B Instruct", "llama"),
    ("Gemma 2 9B IT", "gemma"),
    ("Qwen2.5 7B Instruct", "qwen"),
    ("Ministral 8B Instruct 2410", "ministral"),
]

mode = widgets.ToggleButtons(
    options=[("Quick model", "quick"), ("Paper reproduction", "paper")],
    value="quick",
    description="Run mode:",
    style={"description_width": "initial"},
)
model = widgets.Dropdown(
    options=QUICK_MODELS,
    value="llama",
    description="Base model:",
    style={"description_width": "initial"},
)

def update_model_options(change):
    previous = model.value
    options = QUICK_MODELS if change["new"] == "quick" else PAPER_MODELS
    model.options = options
    available = {value for _, value in options}
    model.value = previous if previous in available else "llama"

mode.observe(update_model_options, names="value")
save_to_drive = widgets.Checkbox(value=True, description="Save checkpoints to Google Drive")
display(mode, model, save_to_drive)

### Optional FP8 setup for paper mode
Run this cell after choosing the mode above. For **Quick model**, it removes leftover Transformer Engine packages from earlier installs and skips FP8 installation. PEFT can otherwise import a broken optional FP8 library even when training uses FP16. Paper mode uses Transformer Engine only on GPUs with supported native FP8 (Ada SM89, Hopper, or Blackwell). On other GPUs the existing paper configuration falls back to BF16 and this install is skipped. The full-size paper models still need a high-memory GPU.

Transformer Engine can compile a CUDA extension and take a long time. `--no-build-isolation` lets the build use the installed PyTorch; it does not eliminate compilation. If you change the run mode or GPU, rerun this cell before training.


In [ ]:
import importlib
import importlib.util
from importlib import metadata
import subprocess, sys, time
import torch
# Refresh helpers if this kernel already imported an older checkout.
from diffusion_lm import notebook as notebook_helpers
importlib.reload(notebook_helpers)
from diffusion_lm.notebook import remove_unused_transformer_engine

if mode.value != "paper":
    remove_unused_transformer_engine()
    print("Quick mode: skipping Transformer Engine; FP8 is disabled.", flush=True)
elif not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime under Runtime > Change runtime type.")
else:
    capability = torch.cuda.get_device_capability()
    native_fp8 = capability == (8, 9) or capability[0] >= 9
    if not native_fp8:
        remove_unused_transformer_engine()
        print("This GPU uses the existing BF16 paper fallback; no FP8 install needed.", flush=True)
    else:
        try:
            installed_te = metadata.version("transformer-engine")
        except metadata.PackageNotFoundError:
            installed_te = None
        if installed_te == "2.13.0" and importlib.util.find_spec("transformer_engine_torch") is not None:
            print("Transformer Engine 2.13.0 is already installed.", flush=True)
        else:
            print("Install optional FP8 support (CUDA extension build may take several minutes)…", flush=True)
            started = time.monotonic()
            subprocess.run([
                sys.executable, "-m", "pip", "install", "--no-build-isolation",
                "transformer-engine[pytorch]==2.13.0",
            ], check=True)
            importlib.invalidate_caches()
            print(f"FP8 setup completed in {time.monotonic() - started:.1f}s", flush=True)


## 4. Train
The display refreshes with loss curves and recent messages. Dataset/model preparation happens before the first loss point appears. Checkpoints and metrics are written throughout the run.

In [ ]:
import importlib
import datetime
import subprocess
import sys
from pathlib import Path
import torch

# Use the downloaded checkout in this running notebook kernel.
for path in (REPOSITORY_DIR, REPOSITORY_DIR / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

# Refresh helpers if this kernel already imported an older checkout.
from diffusion_lm import notebook as notebook_helpers
importlib.reload(notebook_helpers)
from diffusion_lm.notebook import remove_unused_transformer_engine, run_with_dashboard

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime under Runtime > Change runtime type.")
print("GPU:", torch.cuda.get_device_name(0))

# Run this even when the optional FP8 setup cell was skipped. PEFT must not
# encounter a broken leftover Transformer Engine during non-FP8 training.
capability = torch.cuda.get_device_capability()
uses_native_fp8 = mode.value == "paper" and (capability == (8, 9) or capability[0] >= 9)
if not uses_native_fp8:
    remove_unused_transformer_engine()

from train import load_training_config

if save_to_drive.value:
    from google.colab import drive
    drive.mount("/content/drive")
    output_root = Path("/content/drive/MyDrive/BYOD-runs")
else:
    output_root = Path("/content/BYOD-runs")

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
run_dir = output_root / f"{model.value}-{mode.value}-{stamp}"
config = load_training_config(model.value, mode.value, output_dir=str(run_dir))
print("Selected model:", config["display_name"])
print(f"Training {config['max_updates']:,} updates with rank-{config['lora_r']} LoRA.")

command = [
    sys.executable, "train.py",
    "--model", model.value,
    "--mode", mode.value,
    "--output-dir", str(run_dir),
]
run_with_dashboard(command, run_dir, config["max_updates"], refresh_seconds=5)

## 5. Inspect the result
`best/` is the inference-ready adapter selected using validation loss. The resolved configuration records exactly what was run.

In [ ]:
import json

print("Run directory:", run_dir)
print("Best adapter:", run_dir / "best")
if (run_dir / "test_metrics.json").exists():
    print(json.dumps(json.loads((run_dir / "test_metrics.json").read_text()), indent=2))

## Live demos
Try the full-size converted models directly: [Gemma](https://huggingface.co/spaces/Dovelove/byod-gemma-2-9b), [Llama](https://huggingface.co/spaces/Dovelove/byod-llama-3.1-8b), [Qwen](https://huggingface.co/spaces/Dovelove/byod-qwen2.5-7b), and [Ministral](https://huggingface.co/spaces/Dovelove/byod-ministral-8b).